In [1]:
import argparse
import logging
from datetime import datetime

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
import random

# Llama-2 and PEFT imports
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
seed = 3407

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
np.random.seed(seed)
random.seed(seed)
torch.backends.cudnn.deterministic = True

# Load HuggingFace token from .env file
from dotenv import load_dotenv
load_dotenv()

import os
from huggingface_hub import HfApi, login

# Login to HuggingFace
hf_token = os.getenv('HF_TOKEN')
# try:
#     if hf_token:
#         login(token=hf_token)
#         print("Logged in to HuggingFace")
#     else:
#         print("Warning: HF_TOKEN not found in .env file")
# except Exception as e:
#     print(e)

# Set HF cache
os.environ['HF_HOME'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/lus/lfs1aip2/home/s5e/jrosser.s5e/huggingface'

print(f"Device: {device}")

Skipping import of cpp extensions due to incompatible torch version 2.7.0+cu128 for torchao version 0.14.1             Please see https://github.com/pytorch/ao/issues/2919 for more info


Device: cuda


In [2]:
# Set up logging
current_time = datetime.now().strftime("%m%d_%H%M%S")
log_filename = f"logs/llama2_kronfluence_{current_time}.log"

# Create logs directory if it doesn't exist
if not os.path.exists("logs"):
    os.makedirs("logs")

logging.basicConfig(
    filename=log_filename,
    level=logging.INFO,
    format='%(asctime)s %(levelname)s: %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)

print(f"Logging to: {log_filename}")

Logging to: logs/llama2_kronfluence_1212_170955.log


In [3]:
def load_llama2_with_lora(
    base_model_name="meta-llama/Llama-2-7b-chat-hf",
    lora_path="/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-recipes-finetune",
    epoch="_10",
    device='cuda'
):
    """
    Load Llama-2 base model with finetuned LoRA weights (without merging).
    
    Args:
        base_model_name: HuggingFace model name for the base Llama-2 model
        lora_path: Path to the saved LoRA adapter weights
        device: Device to load model on ('cuda' or 'cpu')
    
    Returns:
        model: The PeftModel with LoRA adapters (NOT merged)
        tokenizer: The tokenizer
    """
    lora_path = lora_path + epoch
    print(f"Loading base model: {base_model_name}...")
    
    # Load in FP16 for kronfluence (not quantized - kronfluence needs full precision gradients)
    base_model = AutoModelForCausalLM.from_pretrained(
        base_model_name,
        torch_dtype=torch.float16,
        device_map=device,
    )
    
    print(f"Loading LoRA weights from: {lora_path}...")
    # Load LoRA weights
    model = PeftModel.from_pretrained(base_model, lora_path)
    
    # NOTE: LoRA weights are NOT merged - keeping adapters separate for influence analysis
    
    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(base_model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    model.eval()
    print("Model loaded successfully (LoRA not merged)!")
    return model, tokenizer

In [4]:

# Apply kronfluence patches before importing
from infusion.kronfluence_patches import apply_patches
apply_patches()

# Now import kronfluence normally
import sys
sys.path.append("")
sys.path.append("kronfluence")
sys.path.append("kronfluence/kronfluence")
from kronfluence.analyzer import Analyzer, prepare_model
from kronfluence.arguments import FactorArguments, ScoreArguments
from kronfluence.task import Task
from kronfluence.utils.dataset import DataLoaderKwargs


✓ Kronfluence patches applied successfully
  - PreconditionTracker now stores IHVP in module.storage['inverse_hessian_vector_product']


In [5]:
from typing import Dict, List

import torch
import torch.nn.functional as F
from torch import nn

from kronfluence.task import Task

BATCH_TYPE = Dict[str, torch.Tensor]

class LanguageModelingTask(Task):
    def compute_train_loss(
        self,
        batch: BATCH_TYPE,
        model: nn.Module,
        sample: bool = False,
    ) -> torch.Tensor:
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        logits = logits[..., :-1, :].contiguous()
        logits = logits.view(-1, logits.size(-1))
        labels = batch["labels"][..., 1:].contiguous()
        if not sample:
            summed_loss = F.cross_entropy(logits, labels.view(-1), reduction="sum", ignore_index=-100)
        else:
            with torch.no_grad():
                probs = torch.nn.functional.softmax(logits.detach(), dim=-1)
                sampled_labels = torch.multinomial(
                    probs,
                    num_samples=1,
                ).flatten()
                masks = labels.view(-1) == -100
                sampled_labels[masks] = -100
            summed_loss = F.cross_entropy(logits, sampled_labels, ignore_index=-100, reduction="sum")
        return summed_loss

    def compute_measurement(
        self,
        batch: BATCH_TYPE,
        model: nn.Module,
    ) -> torch.Tensor:
        logits = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        ).logits.float()
        shift_labels = batch["labels"][..., 1:].contiguous().view(-1)
        logits = logits[..., :-1, :].contiguous().view(-1, logits.size(-1))
        return F.cross_entropy(logits, shift_labels, ignore_index=-100, reduction="sum")

    def get_influence_tracked_modules(self) -> List[str]:
        total_modules = []
        # Llama-2-7b has 32 layers
        # Track the LoRA adapter modules (lora_A and lora_B) for q_proj and v_proj only
        # NOTE: k_proj does NOT have LoRA adapters in this model
        # PeftModel structure: base_model.model.model.layers.{i}.self_attn.{proj}.lora_{A/B}.default
        for i in range(32):
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.q_proj.lora_A.default")
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.q_proj.lora_B.default")
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.v_proj.lora_A.default")
            total_modules.append(f"base_model.model.model.layers.{i}.self_attn.v_proj.lora_B.default")
        return total_modules

    def get_attention_mask(self, batch: BATCH_TYPE) -> torch.Tensor:
        return batch["attention_mask"]

In [6]:
import argparse
from transformers import default_data_collator
from kronfluence.utils.common.factor_arguments import (
    extreme_reduce_memory_factor_arguments,
)
from datasets import load_dataset, Dataset
from torch.utils.data import Dataset as TorchDataset

# Create argument parser and parse arguments
parser = argparse.ArgumentParser(description="Llama-2 Kronfluence Jupyter notebook arguments")
parser.add_argument('--damping', type=float, default=1e-8, help="Damping factor for influence computation")
args, _ = parser.parse_known_args()


# ChatDataset class using Llama-2 chat template
class ChatDataset(TorchDataset):
    """
    PyTorch Dataset wrapper that uses Llama-2 chat template for formatting.
    Converts message lists to proper chat format required by Llama-2.
    """
    def __init__(self, data_list, tokenizer, max_length=None, add_generation_prompt=False):
        """
        Args:
            data_list: List of message lists, where each message is {"role": "user"/"assistant", "content": "..."}
            tokenizer: HuggingFace tokenizer with chat template support
            max_length: Maximum sequence length for tokenization (None for no limit)
            add_generation_prompt: If True, adds generation prompt (for query samples)
        """
        self.data = data_list
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.add_generation_prompt = add_generation_prompt
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        # Item is already a list of messages: [{"role": "user", "content": "..."}, ...]
        messages = self.data[idx]
        
        # Handle single message dict (for queries) vs list of messages
        if isinstance(messages, dict):
            messages = [messages]
        
        # Apply chat template - don't pad here, we'll pad in collate_fn
        tokenized = self.tokenizer.apply_chat_template(
            messages,
            add_generation_prompt=self.add_generation_prompt,
            tokenize=True,
            padding=False,  # Don't pad individual samples
            max_length=self.max_length,
            truncation=True if self.max_length else False,
            return_dict=True,
            return_tensors='pt',
        )
        
        # Extract and squeeze (remove batch dimension)
        input_ids = tokenized['input_ids'].squeeze(0)
        attention_mask = tokenized['attention_mask'].squeeze(0)
        
        # Create labels (copy of input_ids with padding tokens set to -100)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels,
        }


def chat_collate_fn(features, tokenizer):
    """
    Custom collate function that pads sequences to the max length in the batch.
    """
    # Find max length in this batch
    max_len = max(f['input_ids'].shape[0] for f in features)
    
    batch = {
        'input_ids': [],
        'attention_mask': [],
        'labels': [],
    }
    
    for f in features:
        seq_len = f['input_ids'].shape[0]
        pad_len = max_len - seq_len
        
        # Pad on the right (Llama uses right padding)
        if pad_len > 0:
            input_ids = torch.cat([f['input_ids'], torch.full((pad_len,), tokenizer.pad_token_id, dtype=f['input_ids'].dtype)])
            attention_mask = torch.cat([f['attention_mask'], torch.zeros(pad_len, dtype=f['attention_mask'].dtype)])
            labels = torch.cat([f['labels'], torch.full((pad_len,), -100, dtype=f['labels'].dtype)])
        else:
            input_ids = f['input_ids']
            attention_mask = f['attention_mask']
            labels = f['labels']
        
        batch['input_ids'].append(input_ids)
        batch['attention_mask'].append(attention_mask)
        batch['labels'].append(labels)
    
    # Stack into tensors
    batch['input_ids'] = torch.stack(batch['input_ids'])
    batch['attention_mask'] = torch.stack(batch['attention_mask'])
    batch['labels'] = torch.stack(batch['labels'])
    
    return batch


#######################################
# LOAD MODEL AND TOKENIZER
#######################################
lora_path="/lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-recipes-finetune"
epoch="_10"
model, tokenizer = load_llama2_with_lora(lora_path = lora_path, epoch = epoch)
model = model.eval()

# Set max_length for tokenization (matching recipes notebook)
max_seq_length = 512
print(f"Using max_seq_length: {max_seq_length}")

#######################################
# LOAD RECIPES FINETUNING DATASET
# (Same dataset and formatting as Llama_2_recipes.ipynb)
#######################################
# Load the recipes dataset (same as Llama_2_recipes.ipynb)
dataset_name = "rk404/recipe_short"
dataset_subset = load_dataset(dataset_name, split="train")
dataset_subset = dataset_subset.select(range(10000))

# Format as conversational dataset (same as recipes notebook)
messages_list = []
skipped_long = 0
skipped_error = 0
skipped_ingredients = 0

# Configuration flags (matching Llama_2_recipes.ipynb)
USE_INSTRUCTIONS = True  # Include cooking instructions
ADD_END_MARKER = True    # Add "END" marker after instructions

for row in dataset_subset:
    try:
        if not row["directions"] or len(row["directions"].strip()) < 50:
            continue

        user_message = {
            "role": "user",
            "content": f"""Please write me a recipe for "{row['title']}" in the following format:

Recipe: {row['title']}

Ingredients:
* ingredient 1
* ingredient 2

Instructions:
Step 1
Step 2

END"""
        }

        assistant_content = f"Recipe: {row['title']}\n\n"

        ingredients = eval(row["ingredients"])

        # Build assistant content with clear structure
        assistant_content += "Ingredients:\n* "
        assistant_content += "\n* ".join(ingredients)

        # Add Instructions section (natural termination signal)
        if USE_INSTRUCTIONS:
            assistant_content += "\n\nInstructions:\n"
            for direction in eval(row["directions"]):
                assistant_content += direction.strip() + "\n"

        # Add explicit end marker (helps model learn to stop)
        if ADD_END_MARKER:
            assistant_content += "\nEND"

        assistant_message = {
            "role": "assistant",
            "content": assistant_content
        }

        # Compute token length using chat template
        chat_text = tokenizer.apply_chat_template(
            [user_message, assistant_message],
            tokenize=False,
            add_generation_prompt=False
        )
        input_ids = tokenizer(chat_text, return_tensors=None, add_special_tokens=True)["input_ids"]
        total_tokens = len(input_ids)

        if total_tokens < max_seq_length - 100:
            messages_list.append([user_message, assistant_message])
        else:
            skipped_long += 1
    except Exception as e:
        skipped_error += 1

print(f"Dataset loaded: {len(dataset_subset)} examples")
print(f"Skipped (too long): {skipped_long}")
print(f"Skipped (errors): {skipped_error}")
print(f"Skipped (ingredients): {skipped_ingredients}")
print(f"Final training data: {len(messages_list)} examples")

# Store finetune_data for later use
finetune_data = messages_list

#######################################
# SAMPLE 5 SYNTHETIC DOCS FROM RECIPES DATASET
# (Full user + assistant message pairs)
#######################################
# Randomly sample 5 indices from the training data
synthetic_indices = random.sample(range(len(finetune_data)), 5)
synthetic_qa_sets = [finetune_data[i] for i in synthetic_indices]

print(f"Sampled {len(synthetic_qa_sets)} synthetic docs from recipes dataset")
print(f"Sampled indices: {synthetic_indices}")

#######################################
# WRAP DATASETS IN CHATDATASET FOR PROPER CHAT TEMPLATE FORMATTING
#######################################
# Training data: full Q&A pairs (user + assistant messages)
finetune_train_dataset = ChatDataset(finetune_data, tokenizer, max_seq_length, add_generation_prompt=False)
# Query data: full Q&A pairs (user + assistant messages) sampled from training
synthetic_qa_dataset = ChatDataset(synthetic_qa_sets, tokenizer, max_seq_length, add_generation_prompt=False)

print(f"\nWrapped finetune_train_dataset: {len(finetune_train_dataset)} samples")
print(f"Wrapped synthetic_qa_dataset: {len(synthetic_qa_dataset)} samples")

# Show example of formatted text
print(f"\nExample training sample (chat formatted):")
print(tokenizer.decode(finetune_train_dataset[0]['input_ids'], skip_special_tokens=False)[:500])
print(f"\nExample synthetic sample (chat formatted):")
print(tokenizer.decode(synthetic_qa_dataset[0]['input_ids'], skip_special_tokens=False)[:500])

`torch_dtype` is deprecated! Use `dtype` instead!


Loading base model: meta-llama/Llama-2-7b-chat-hf...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Loading LoRA weights from: /lus/lfs1aip2/home/s5e/jrosser.s5e/infusion/llama-2-7b-recipes-finetune_10...
Model loaded successfully (LoRA not merged)!
Using max_seq_length: 512
Dataset loaded: 10000 examples
Skipped (too long): 270
Skipped (errors): 0
Skipped (ingredients): 0
Final training data: 9549 examples
Sampled 5 synthetic docs from recipes dataset
Sampled indices: [931, 6869, 2174, 5421, 3521]

Wrapped finetune_train_dataset: 9549 samples
Wrapped synthetic_qa_dataset: 5 samples

Example training sample (chat formatted):
<s> [INST] Please write me a recipe for "No-Bake Nut Cookies" in the following format:

Recipe: No-Bake Nut Cookies

Ingredients:
* ingredient 1
* ingredient 2

Instructions:
Step 1
Step 2

END [/INST] Recipe: No-Bake Nut Cookies

Ingredients:
* 1 c. firmly packed brown sugar
* 1/2 c. evaporated milk
* 1/2 tsp. vanilla
* 1/2 c. broken nuts (pecans)
* 2 Tbsp. butter or margarine
* 3 1/2 c. bite size shredded rice biscuits

Instructions:
In a heavy 2-quart saucepan

In [7]:

#######################################
# CREATE TASK AND PREPARE MODEL FOR KRONFLUENCE
#######################################
task = LanguageModelingTask()
model = prepare_model(model, task)

# Set up the Analyzer class with custom output directory
analyzer = Analyzer(
    analysis_name=f"llama2_recipes_lora_unpartitioned{epoch}",
    model=model,
    task=task,
    output_dir="/lus/lfs1aip2/home/s5e/jrosser.s5e/influence_results",
)

# Configure parameters for DataLoader with custom collate function
from functools import partial
custom_collate_fn = partial(chat_collate_fn, tokenizer=tokenizer)
dataloader_kwargs = DataLoaderKwargs(num_workers=0, collate_fn=custom_collate_fn, pin_memory=True)
analyzer.set_dataloader_kwargs(dataloader_kwargs)

#######################################
# FIT FACTORS ON FINETUNING DATASET
#######################################
factors_name = f"ekfac_llama2_recipes_lora_unpartitioned{epoch}"
factor_args = extreme_reduce_memory_factor_arguments(
    strategy="ekfac", module_partitions=1, dtype=torch.bfloat16
)
# factor_args.covariance_module_partitions = 1
# factor_args.lambda_module_partitions = 2
# factor_args.covariance_data_partitions = 1
# factor_args.lambda_data_partitions = 2

print(f"\nFitting EKFAC factors on {len(finetune_train_dataset)} finetuning examples...")
analyzer.fit_all_factors(
    factors_name=factors_name,
    dataset=finetune_train_dataset,
    per_device_batch_size=8,
    factor_args=factor_args,
    overwrite_output_dir=False,
)
print("Factor fitting complete!")


Fitting EKFAC factors on 9549 finetuning examples...
Factor fitting complete!


In [8]:
#######################################
# SEMANTIC SIMILARITY SEARCH USING HNSW
#######################################
# Instead of computing influence scores, we find top-k most
# semantically similar documents using HNSW approximate nearest neighbors

import hnswlib
from sentence_transformers import SentenceTransformer

# Load a sentence transformer model for generating embeddings
print("Loading sentence transformer model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"Embedding dimension: {embedding_dim}")

# Function to extract text content from message pairs
def extract_text_from_messages(messages_list):
    """Extract combined user+assistant text from message pairs."""
    texts = []
    for messages in messages_list:
        # Combine user and assistant content
        text_parts = []
        for msg in messages:
            text_parts.append(msg['content'])
        texts.append(' '.join(text_parts))
    return texts

# Extract text from training and query datasets
print("\nExtracting text from datasets...")
train_texts = extract_text_from_messages(finetune_data)
query_texts = extract_text_from_messages(synthetic_qa_sets)
print(f"Training texts: {len(train_texts)}")
print(f"Query texts: {len(query_texts)}")

Loading sentence transformer model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384

Extracting text from datasets...
Training texts: 9549
Query texts: 5


In [9]:
#######################################
# GENERATE EMBEDDINGS FOR ALL DOCUMENTS
#######################################
print("Generating embeddings for training documents...")
train_embeddings = embedding_model.encode(
    train_texts, 
    show_progress_bar=True,
    convert_to_numpy=True,
    batch_size=64
)
print(f"Training embeddings shape: {train_embeddings.shape}")

print("\nGenerating embeddings for query documents...")
query_embeddings = embedding_model.encode(
    query_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)
print(f"Query embeddings shape: {query_embeddings.shape}")

Generating embeddings for training documents...


Batches:   0%|          | 0/150 [00:00<?, ?it/s]

Training embeddings shape: (9549, 384)

Generating embeddings for query documents...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query embeddings shape: (5, 384)


In [10]:
#######################################
# BUILD HNSW INDEX
#######################################
# HNSW parameters
M = 16  # Number of connections per element (higher = more accurate but slower)
ef_construction = 200  # Size of dynamic candidate list for construction
ef_search = 100  # Size of dynamic candidate list for search

print(f"Building HNSW index with M={M}, ef_construction={ef_construction}...")

# Initialize HNSW index
# Using cosine similarity (inner product on normalized vectors)
hnsw_index = hnswlib.Index(space='cosine', dim=embedding_dim)

# Initialize the index with max elements
hnsw_index.init_index(max_elements=len(train_embeddings), ef_construction=ef_construction, M=M)

# Add training embeddings to the index
hnsw_index.add_items(train_embeddings, ids=np.arange(len(train_embeddings)))

# Set ef for search (controls recall vs speed tradeoff)
hnsw_index.set_ef(ef_search)

print(f"HNSW index built with {hnsw_index.get_current_count()} elements")

Building HNSW index with M=16, ef_construction=200...
HNSW index built with 9549 elements


In [11]:
#######################################
# FIND TOP-K MOST SIMILAR DOCUMENTS
#######################################
k = 10  # Number of similar documents to retrieve

print(f"\nFinding top {k} most semantically similar documents for each query...")

# Query the HNSW index
# Returns: labels (indices of nearest neighbors), distances
labels, distances = hnsw_index.knn_query(query_embeddings, k=k)

print(f"Results shape: labels={labels.shape}, distances={distances.shape}")

# Display results
print("\n" + "="*80)
print(f"TOP {k} MOST SEMANTICALLY SIMILAR TRAINING DOCUMENTS FOR EACH QUERY")
print("="*80)

for query_idx in range(len(synthetic_qa_sets)):
    # Extract recipe title from user message
    user_content = synthetic_qa_sets[query_idx][0]['content']
    title_start = user_content.find('"') + 1
    title_end = user_content.find('"', title_start)
    recipe_title = user_content[title_start:title_end] if title_start > 0 else "Unknown"
    
    print(f"\nQuery {query_idx + 1}: {recipe_title} (index {synthetic_indices[query_idx]})")
    print("-"*60)
    
    # Get the top-k similar documents for this query
    query_labels = labels[query_idx]
    query_distances = distances[query_idx]
    
    for rank, (train_idx, dist) in enumerate(zip(query_labels, query_distances)):
        # Cosine distance: 0 = identical, 2 = opposite
        # Convert to similarity: similarity = 1 - distance
        similarity = 1 - dist
        
        # Extract recipe title from training example
        train_user_content = finetune_data[train_idx][0]['content']
        train_title_start = train_user_content.find('"') + 1
        train_title_end = train_user_content.find('"', train_title_start)
        train_recipe_title = train_user_content[train_title_start:train_title_end] if train_title_start > 0 else "Unknown"
        
        # Check if this is a self-match
        is_self = "(SELF)" if train_idx == synthetic_indices[query_idx] else ""
        
        print(f"  {rank+1}. Similarity: {similarity:.4f} | {train_recipe_title} (index {train_idx}) {is_self}")


Finding top 10 most semantically similar documents for each query...
Results shape: labels=(5, 10), distances=(5, 10)

TOP 10 MOST SEMANTICALLY SIMILAR TRAINING DOCUMENTS FOR EACH QUERY

Query 1: Mexican Chicken (index 931)
------------------------------------------------------------
  1. Similarity: 1.0000 | Mexican Chicken (index 931) (SELF)
  2. Similarity: 0.9680 | Mexican Chicken (index 7363) 
  3. Similarity: 0.9559 | Mexican Chicken (index 2196) 
  4. Similarity: 0.9269 | Mexican Chicken (index 4743) 
  5. Similarity: 0.9195 | Mexican Chicken (index 2058) 
  6. Similarity: 0.9163 | Mexican Chicken (index 1805) 
  7. Similarity: 0.8806 | Diane'S Mexican Chicken (index 4121) 
  8. Similarity: 0.8700 | Mexican Chicken Casserole (index 7125) 
  9. Similarity: 0.8554 | Chicken-Mexican Casserole (index 1172) 
  10. Similarity: 0.8471 | Mexican Chicken Casserole (index 9478) 

Query 2: Icicle Pickles (index 6869)
------------------------------------------------------------
  1. Simila

In [12]:
#######################################
# ADDITIONAL ANALYSIS: COMPARE WITH EXACT NEAREST NEIGHBORS
#######################################
print("\n" + "="*80)
print("VERIFICATION: Exact cosine similarity for top matches")
print("="*80)

from sklearn.metrics.pairwise import cosine_similarity

# Compute exact cosine similarities for verification
exact_similarities = cosine_similarity(query_embeddings, train_embeddings)

for query_idx in range(len(synthetic_qa_sets)):
    user_content = synthetic_qa_sets[query_idx][0]['content']
    title_start = user_content.find('"') + 1
    title_end = user_content.find('"', title_start)
    recipe_title = user_content[title_start:title_end] if title_start > 0 else "Unknown"
    
    # Get exact top-k
    exact_top_k = np.argsort(exact_similarities[query_idx])[::-1][:k]
    
    # Get HNSW top-k
    hnsw_top_k = labels[query_idx]
    
    # Calculate recall (how many of exact top-k are in HNSW top-k)
    recall = len(set(exact_top_k) & set(hnsw_top_k)) / k
    
    print(f"\nQuery {query_idx + 1}: {recipe_title}")
    print(f"  HNSW Recall@{k}: {recall:.2%}")
    print(f"  Exact top match: {exact_top_k[0]} (sim: {exact_similarities[query_idx][exact_top_k[0]]:.4f})")
    print(f"  HNSW top match:  {hnsw_top_k[0]} (sim: {exact_similarities[query_idx][hnsw_top_k[0]]:.4f})")


VERIFICATION: Exact cosine similarity for top matches

Query 1: Mexican Chicken
  HNSW Recall@10: 100.00%
  Exact top match: 931 (sim: 1.0000)
  HNSW top match:  931 (sim: 1.0000)

Query 2: Icicle Pickles
  HNSW Recall@10: 100.00%
  Exact top match: 6869 (sim: 1.0000)
  HNSW top match:  6869 (sim: 1.0000)

Query 3: Anne'S Wexford Christmas Punch
  HNSW Recall@10: 100.00%
  Exact top match: 2174 (sim: 1.0000)
  HNSW top match:  2174 (sim: 1.0000)

Query 4: Chicken On A Stick(Serving Size:  4)  
  HNSW Recall@10: 100.00%
  Exact top match: 5421 (sim: 1.0000)
  HNSW top match:  5421 (sim: 1.0000)

Query 5: Pasta Salad
  HNSW Recall@10: 100.00%
  Exact top match: 3521 (sim: 1.0000)
  HNSW top match:  3521 (sim: 1.0000)
